In [1]:
from datasets import load_dataset

ds = load_dataset("bearberry/rus_xquadqa")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

rus_xquadqa.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1190 [00:00<?, ? examples/s]

In [2]:
print(ds.keys())

dict_keys(['train'])


In [3]:
import pandas as pd

df = ds['train'].to_pandas()
print(df.head())
print(df.columns)

           id                                           question  \
0  0_rusxquad             Сколько очков уступила защита Пэнтерс?   
1  1_rusxquad   Сколько мешков за карьеру было у Джареда Аллена?   
2  2_rusxquad  Сколько блокировок записал на свой счет Люк Ки...   
3  3_rusxquad              Сколько мячей перехватил Джош Норман?   
4  4_rusxquad  Кто больше записал на свой счет мешков в коман...   

         answers normalized_answers  \
0          [308]              [308]   
1          [136]              [136]   
2          [118]              [118]   
3    [4, четыре]        [четыре, 4]   
4  [Кейван Шорт]     [кейван шорты]   

                                             context  \
0  [{'chunk': 'Защита Пэнтерс уступила всего 308 ...   
1  [{'chunk': 'Защита Пэнтерс уступила всего 308 ...   
2  [{'chunk': 'Защита Пэнтерс уступила всего 308 ...   
3  [{'chunk': 'Защита Пэнтерс уступила всего 308 ...   
4  [{'chunk': 'Защита Пэнтерс уступила всего 308 ...   

                   

In [4]:
print(df.tail())

                 id                                           question  \
1185  1185_rusxquad  Какая величина может быть записана как сумма э...   
1186  1186_rusxquad            Что вызывает напряжение в конструкциях?   
1187  1187_rusxquad  Какая величина используется для расчета площад...   
1188  1188_rusxquad  Что связано с силами, направленными перпендику...   
1189  1189_rusxquad  Что включает в себя компоненты давления при ра...   

                                answers                 normalized_answers  \
1185          [электростатической силы]          [электростатический сила]   
1186                [Тензор напряжений]                [тензор напряжение]   
1187              [компоненты давления]               [компонент давление]   
1188              [компоненты давления]               [компонент давление]   
1189  [математические формулы, формулы]  [математический формула, формула]   

                                                context  \
1185  [{'chunk': 'Определяя

In [5]:
df['context'][0]

array([{'chunk': 'Защита Пэнтерс уступила всего 308 очков, заняв шестое место в лиге, а также лидировала в НФЛ по перехватам с 24 и похвасталась четырьмя попаданиями в Пробоул.', 'is_relevant': True},
       {'chunk': 'Дифенсив тэкл Пробоула Кейван Шорт лидирует в команде с 11 мешками, а также обеспечил три потери мяча и получил два.', 'is_relevant': False},
       {'chunk': 'Нападающий Марио Эдисон добавил 61⁄2 мешков.', 'is_relevant': False},
       {'chunk': 'Линия Пэнтерс также представила ди-энда-ветерана Джареда Аллена, пятикратного участника Пробоула, который был активным лидером по количеству мешков в карьере НФЛ в количестве 136, вместе с ди-эндом Кони Или, у которого было 5 мешков всего за 9 стартов.', 'is_relevant': False},
       {'chunk': 'Позади них для участия в Пробоуле также были выбраны два из трех стартовых лайнбекеров Пэнтерс: Томас Дэвис и Люк Кикли.', 'is_relevant': False},
       {'chunk': 'Дэвис собрал 51⁄2 мешков, четыре вынужденных потери мяча и четыре перехва

In [6]:
df['answers'][0]

array(['308'], dtype=object)

In [7]:
df['question'][0]

'Сколько очков уступила защита Пэнтерс?'

In [8]:
df_exp = df.explode('context')

df_exp['chunk_text'] = df_exp['context'].apply(lambda x: x['chunk'])

corpus = df_exp['chunk_text'].drop_duplicates().tolist()

print(f"Всего уникальных фрагментов: {len(corpus)}")

Всего уникальных фрагментов: 1237


In [9]:
corpus_set = set()
for example in ds['train']:
    for ctx in example['context']:
        corpus_set.add(ctx['chunk'])
corpus_2 = list(corpus_set)
print(f"Всего уникальных фрагментов: {len(corpus_2)}")

Всего уникальных фрагментов: 1237



Немного смутило число 1237, решил проверить другим способом


In [10]:
!nvidia-smi

Sun Mar 22 20:15:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:
!pip install -q -U bitsandbytes
!pip install -q -U transformers
!pip install -q -U peft
!pip install -q -U accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 95.6 MB/s eta 0:00:00


In [12]:
!pip install qdrant-client==1.7.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.4 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you

In [13]:
!pip install torch transformers

In [14]:
import torch
from transformers import AutoTokenizer, AutoModel

model_name = "intfloat/multilingual-e5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings(
    (word_embeddings): Embedding(250002, 1024, padding_idx=1)
    (token_type_embeddings): Embedding(1, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 1024, padding_idx=1)
  )
  (encoder): XLMRobertaEncoder(
    (layer): ModuleList(
      (0-23): 24 x XLMRobertaLayer(
        (attention): XLMRobertaAttention(
          (self): XLMRobertaSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): XLMRobertaSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwi

In [15]:
def get_embedding(texts, prefix="passage: "):
    texts = [prefix + t for t in texts]
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors='pt')
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)

    attention_mask = inputs['attention_mask']
    token_embeddings = outputs.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sentence_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    sentence_embeddings = torch.nn.functional.normalize(sentence_embeddings, p=2, dim=1)
    return sentence_embeddings.cpu().numpy()

In [16]:
from qdrant_client import QdrantClient
from qdrant_client.http import models
from qdrant_client.models import PointStruct
import numpy as np
from tqdm import tqdm

client = QdrantClient(":memory:")

vector_size = 1024
collection_name = "rus_xquad_corpus"
client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=vector_size,
        distance=models.Distance.COSINE
    )
)

batch_size = 32
total = len(corpus)

for i in tqdm(range(0, total, batch_size)):
    batch_texts = corpus[i:i+batch_size]
    vectors = get_embedding(batch_texts)

    vectors_list = vectors.tolist()
    ids = list(range(i, i+len(batch_texts)))

    points = [
        PointStruct(
            id=idx,
            vector=vec,
            payload={"text": text}
        )
        for idx, vec, text in zip(ids, vectors_list, batch_texts)
    ]
    client.upsert(
        collection_name=collection_name,
        points=points
    )

print(f"Индексация завершена. Всего точек: {client.count(collection_name=collection_name).count}")

100%|██████████| 39/39 [00:23<00:00,  1.66it/s]

Индексация завершена. Всего точек: 1237


In [17]:
question = ds['train'][0]['question']
print("Вопрос:", question)

query_vec = get_embedding([f"query: {question}"])[0]

search_results = client.search(
    collection_name=collection_name,
    query_vector=query_vec.tolist(),
    limit=5,
    with_payload=True
)

for res in search_results:
    print(f"Score: {res.score:.4f} | Text: {res.payload['text'][:100]}...")

Вопрос: Сколько очков уступила защита Пэнтерс?
Score: 0.8720 | Text: Защита Пэнтерс уступила всего 308 очков, заняв шестое место в лиге, а также лидировала в НФЛ по пере...
Score: 0.8148 | Text: Дэвис собрал 51⁄2 мешков, четыре вынужденных потери мяча и четыре перехвата, в то время как Кикли ли...
Score: 0.8104 | Text: д.....
Score: 0.8090 | Text: Дифенсив тэкл Пробоула Кейван Шорт лидирует в команде с 11 мешками, а также обеспечил три потери мяч...
Score: 0.8082 | Text: Название происходит от греческого корня ????...


In [18]:
!pip install -q -U langchain langchain-community
# qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatibl

In [19]:
# !pip install qdrant-client==1.7.3

In [20]:
from langchain_community.vectorstores import Qdrant
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")
doc_store = Qdrant(
    client,
    collection_name="rus_xquad_corpus",
    embeddings=embeddings
    , content_payload_key='text'
)

retriever = doc_store.as_retriever(search_type='similarity', search_kwargs={'k':10})
docs = retriever.invoke('query: ' + question)

/tmp/ipykernel_11212/3601176260.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

/tmp/ipykernel_11212/3601176260.py:5: LangChainDeprecationWarning: The class `Qdrant` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-qdrant package and should be used instead. To use it run `pip install -U `langchain-qdrant` and import as `from `langchain_qdrant import Qdrant``.
  doc_store = Qdrant(


In [21]:
docs

[Document(metadata={'_id': 0, '_collection_name': 'rus_xquad_corpus'}, page_content='Защита Пэнтерс уступила всего 308 очков, заняв шестое место в лиге, а также лидировала в НФЛ по перехватам с 24 и похвасталась четырьмя попаданиями в Пробоул.'),
 Document(metadata={'_id': 5, '_collection_name': 'rus_xquad_corpus'}, page_content='Дэвис собрал 51⁄2 мешков, четыре вынужденных потери мяча и четыре перехвата, в то время как Кикли лидировал в команде по блокировкам (118), форсировал две потери мяча и перехватил четыре своих передачи.'),
 Document(metadata={'_id': 3, '_collection_name': 'rus_xquad_corpus'}, page_content='Линия Пэнтерс также представила ди-энда-ветерана Джареда Аллена, пятикратного участника Пробоула, который был активным лидером по количеству мешков в карьере НФЛ в количестве 136, вместе с ди-эндом Кони Или, у которого было 5 мешков всего за 9 стартов.'),
 Document(metadata={'_id': 18, '_collection_name': 'rus_xquad_corpus'}, page_content='Затем Андерсон забил гол после пере

In [22]:
for res in docs:
    print(f"Text: {res.page_content[:100]}...")

Text: Защита Пэнтерс уступила всего 308 очков, заняв шестое место в лиге, а также лидировала в НФЛ по пере...
Text: Дэвис собрал 51⁄2 мешков, четыре вынужденных потери мяча и четыре перехвата, в то время как Кикли ли...
Text: Линия Пэнтерс также представила ди-энда-ветерана Джареда Аллена, пятикратного участника Пробоула, ко...
Text: Затем Андерсон забил гол после перехвата с 2-ярдовом тачдауном, и Мэннинг завершил пас Бенни Фаулеру...
Text: Дифенсив тэкл Пробоула Кейван Шорт лидирует в команде с 11 мешками, а также обеспечил три потери мяч...
Text: Тем временем, нападение Денвера удерживалось за пределами зоны защиты в течение трех игр, но штрафно...
Text: Затем они победили действующего чемпиона Суперкубка XLIX Нью-Ингленд Пэтриотс в игре чемпионата АФК,...
Text: Бронкос победил Питтсбург Стилерс в дивизионном раунде, 23–16, набрав 11 очков в последние три минут...
Text: Позади них для участия в Пробоуле также были выбраны два из трех стартовых лайнбекеров Пэнтерс: Тома...
Text: В сл

In [23]:
import os
import sys
import torch
from transformers import AutoTokenizer, GenerationConfig, AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig

model_name = 'RefalMachine/RuadaptQwen3-4B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

modelLM = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    #quantization_config=bnb_config, # Раскомментировать в случае использования моделей побольше, где не нужна квантизация
    device_map="cuda:0", # В случае нескольких GPU нужно указать конкретную, cuda:0, например
    attn_implementation="sdpa", #attn_implementation='flash_attention_2' для GPU, которые поддерживают
)
modelLM.eval()

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.4M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(146260, 2560, padding_idx=146213)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (

In [24]:
from transformers import GenerationConfig

generation_config = GenerationConfig.from_pretrained(model_name)
generation_config

GenerationConfig {
  "do_sample": true,
  "eos_token_id": 146215,
  "pad_token_id": 146213,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}

In [25]:
generation_config.temperature = 0.3
generation_config.max_new_tokens = 512
generation_config

GenerationConfig {
  "do_sample": true,
  "eos_token_id": 146215,
  "max_new_tokens": 512,
  "pad_token_id": 146213,
  "temperature": 0.3,
  "top_k": 20,
  "top_p": 0.8
}

In [26]:
input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': 'Что такое ИИ?'}
], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
input_ids = input_ids.to(modelLM.device)

with torch.no_grad():
    output = modelLM.generate(
        input_ids,
        generation_config=generation_config
    )[:, input_ids.shape[-1]:].detach().cpu()

output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
output

'**ИИ — это Искусственный интеллект** (от англ. *Artificial Intelligence*), то есть система, способная выполнять задачи, требующие от человека интеллекта. Такие задачи включают:\n\n- Обработку естественного языка (понимание и генерация текста, чат-боты),\n- Распознавание образов (например, распознавание лиц, анализ изображений),\n- Решение логических задач,\n- Обучение на основе опыта (машинное обучение),\n- Принятие решений в условиях неопределённости,\n- Планирование и прогнозирование.\n\nИИ — это область компьютерных наук, где разрабатываются алгоритмы и модели, способные имитировать человеческие способности: мышление, обучение, понимание, принятие решений и даже творчество.\n\n### Примеры ИИ:\n- Чат-боты (например, Siri, Alexa, Google Assistant),\n- Рекомендательные системы (YouTube, Netflix, Spotify),\n- Автодороги с автономным вождением,\n- Диагностика болезней на основе медицинских изображений,\n- Генерация музыки, картин, текстов (например, DALL·E, Midjourney, ChatGPT).\n\n### 

In [27]:
print(output)

**ИИ — это Искусственный интеллект** (от англ. *Artificial Intelligence*), то есть система, способная выполнять задачи, требующие от человека интеллекта. Такие задачи включают:

- Обработку естественного языка (понимание и генерация текста, чат-боты),
- Распознавание образов (например, распознавание лиц, анализ изображений),
- Решение логических задач,
- Обучение на основе опыта (машинное обучение),
- Принятие решений в условиях неопределённости,
- Планирование и прогнозирование.

ИИ — это область компьютерных наук, где разрабатываются алгоритмы и модели, способные имитировать человеческие способности: мышление, обучение, понимание, принятие решений и даже творчество.

### Примеры ИИ:
- Чат-боты (например, Siri, Alexa, Google Assistant),
- Рекомендательные системы (YouTube, Netflix, Spotify),
- Автодороги с автономным вождением,
- Диагностика болезней на основе медицинских изображений,
- Генерация музыки, картин, текстов (например, DALL·E, Midjourney, ChatGPT).

### Важно:
ИИ **не раве

In [28]:
def help_f(c, q):
  prompt = f"Вопрос: {q}\n\nКонтекст:\n{c}\n\nОтвет одним словом:"
  input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': prompt}
  ], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
  input_ids = input_ids.to(modelLM.device)

  with torch.no_grad():
      output = modelLM.generate(
          input_ids,
          generation_config=generation_config
      )[:, input_ids.shape[-1]:].detach().cpu()

  output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
  return output

In [29]:
for i in range(10):
  question = df['question'][i]
  answer = df['answers'][i]
  docs = retriever.invoke('query: ' + question)
  context = "\n\n".join([doc.page_content for doc in docs])
  out = help_f(context, question)
  print(question)
  print(f"Правильный ответ - {answer}")
  print(f"Ответ модели - {out}")
  print(answer==out)
  print("="*30)


Сколько очков уступила защита Пэнтерс?
Правильный ответ - ['308']
Ответ модели - 308
[ True]
Сколько мешков за карьеру было у Джареда Аллена?
Правильный ответ - ['136']
Ответ модели - 136
[ True]
Сколько блокировок записал на свой счет Люк Кикли?
Правильный ответ - ['118']
Ответ модели - 118
[ True]
Сколько мячей перехватил Джош Норман?
Правильный ответ - ['4' 'четыре']
Ответ модели - четыре
[False  True]
Кто больше записал на свой счет мешков в команде в этом сезоне?
Правильный ответ - ['Кейван Шорт']
Ответ модели - Мэннинг
[False]
Сколько перехватов приписывают защите Пэнтерс в 2015 году?
Правильный ответ - ['24']
Ответ модели - 24
[ True]
Кто был лидером Пэнтерс по мешкам?
Правильный ответ - ['Кейван Шорт']
Ответ модели - Джаред Аллен
[False]
Сколько игроков защиты Пэнтерс было выбрано для Пробоула?
Правильный ответ - ['4' 'четыре']
Ответ модели - Семь
[False False]
Сколько вынужденных потерь мяча имел Томас Дэвис?
Правильный ответ - ['4' 'четыре']
Ответ модели - четыре
[False  True

In [30]:
def help_f(c, q):
  prompt = f"Вопрос: {q}\n\nпо этому вопросу найденны документы, в которых есть ответ:\n{c}\n\n Можешь подумать и запиши в конце ответ одним словом:"
  input_ids = tokenizer.apply_chat_template([
    {'role': 'user', 'content': prompt}
  ], add_generation_prompt=True, tokenize=True, return_tensors='pt')['input_ids']
  input_ids = input_ids.to(modelLM.device)

  with torch.no_grad():
      output = modelLM.generate(
          input_ids,
          generation_config=generation_config
      )[:, input_ids.shape[-1]:].detach().cpu()

  output = tokenizer.batch_decode(output, skip_special_tokens=True)[0]
  return output

for i in range(10):
  question = df['question'][i]
  answer = df['answers'][i]
  docs = retriever.invoke('query: ' + question)
  context = "\n\n".join([doc.page_content for doc in docs])
  out = help_f(context, question)
  print(question)
  print(f"Правильный ответ - {answer}")
  print(f"Ответ модели - {out}")
  print(answer==(out.split()[-1]))
  print("="*30)

Сколько очков уступила защита Пэнтерс?
Правильный ответ - ['308']
Ответ модели - Защита Пэнтерс уступила 308 очков.

**Ответ: 308**
[False]
Сколько мешков за карьеру было у Джареда Аллена?
Правильный ответ - ['136']
Ответ модели - Джаред Аллен набрал 136 мешков за карьеру в НФЛ.
[False]
Сколько блокировок записал на свой счет Люк Кикли?
Правильный ответ - ['118']
Ответ модели - Люк Кикли записал на свой счёт **118** блокировок.

Ответ: **118**
[False]
Сколько мячей перехватил Джош Норман?
Правильный ответ - ['4' 'четыре']
Ответ модели - Джош Норман перехватил **четыре** мяча.

Ответ: четыре
[False  True]
Кто больше записал на свой счет мешков в команде в этом сезоне?
Правильный ответ - ['Кейван Шорт']
Ответ модели - Марио Эдисон

(Он записал 61⁄2 мешков, что больше, чем у всех других упомянутых игроков.)
[False]
Сколько перехватов приписывают защите Пэнтерс в 2015 году?
Правильный ответ - ['24']
Ответ модели - Защита Пэнтерс в 2015 году приписала **24 перехвата**.

Одно слово: **24**
[